# Semana 31 — RFCs, ADRs e Reproducibility com um Comando
**Fase 7: Governance e Engineering at Scale** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/07-Phase-7-Governance-and-Scale/notebooks/semana_31_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-31-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/07-Phase-7-Governance-and-Scale/docs/Semana-31-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_31_gabarito.ipynb`.

In [ ]:
import json, hashlib, sys, subprocess, platform, importlib.metadata as md
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pydantic import BaseModel, Field
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

## Parte A — Reproducibility

### Passo 1 — O problema
O mesmo "treino" executado duas vezes, sem seeds.

In [ ]:
def treino_descuidado():
    X, y = make_classification(n_samples=4000, weights=[0.95], random_state=0)
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3)                 # sem random_state
    m = RandomForestClassifier(n_estimators=30).fit(Xa, ya)                # sem random_state
    return round(recall_score(yb, m.predict(Xb)), 4)
print("execução 1:", treino_descuidado(), "| execução 2:", treino_descuidado())

Qual das duas vai para o relatório? Qual está "em produção"?

### Passo 2 — Configuração como objeto
Tudo o que influencia o resultado entra em um objeto validado. Nada de valores soltos no código.

In [ ]:
# [SEU CÓDIGO AQUI] defina Config com: n_amostras, proporcao_teste, n_estimators, class_weight, limiar, semente
cfg = Config(class_weight="balanced_subsample", limiar=0.3); print(cfg.model_dump())

### Passo 3 — Impressão digital dos dados
`hash_array` devolve os 16 primeiros caracteres do SHA-256 dos bytes de um array (ou de vários, concatenados).

In [ ]:
def hash_array(*arrays):
    # [SEU CÓDIGO AQUI] SHA-256 dos bytes dos arrays, 16 caracteres

a = np.arange(10.0); assert hash_array(a) == hash_array(a.copy()) and hash_array(a) != hash_array(a + 1e-9); print(hash_array(a))

### Passo 4 — Capturar o ambiente
Versão do Python, sistema, versões das bibliotecas relevantes e o commit do git (ou `"desconhecido"` fora de um repositório).

In [ ]:
def ambiente(pacotes=("numpy", "pandas", "scikit-learn", "pydantic")):
    # [SEU CÓDIGO AQUI] devolva dict com python, sistema, commit e pacotes {nome: versão}
print(ambiente())

### Passo 5 — O pipeline que grava um manifest
`executar(cfg)` gera os dados, treina, avalia e devolve um manifest com configuração, hash dos dados, ambiente, métricas e hash das previsões. A seed da configuração deve chegar a **todos** os pontos de aleatoriedade.

In [ ]:
def executar(cfg: Config) -> dict:
    # [SEU CÓDIGO AQUI] gere os dados, divida e treine, usando cfg.semente em todos os lugares
    proba = modelo.predict_proba(Xb)[:, 1]; pred = (proba >= cfg.limiar).astype(int)
    return {"config": cfg.model_dump(), "hash_dados": hash_array(X, y), "ambiente": ambiente(),
            "metricas": {"recall": round(float(recall_score(yb, pred)), 6), "precisao": round(float(precision_score(yb, pred, zero_division=0)), 6)},
            "hash_previsoes": hash_array(np.round(proba, 8))}

m1 = executar(cfg); json.dump(m1, open("manifesto.json", "w"), indent=1); print(json.dumps(m1, indent=1)[:600])

### Passo 6 — Reproduzir e conferir
`reproduzir(caminho)` lê o manifest, reexecuta com a mesma configuração e compara hash dos dados, métricas e hash das previsões. Devolve um dicionário de verificações booleanas.

In [ ]:
def reproduzir(caminho="manifesto.json"):
    original = json.load(open(caminho))
    # [SEU CÓDIGO AQUI] reexecute e compare os quatro itens

verificacao = reproduzir(); print(verificacao); assert all(verificacao.values())

### Passo 7 — Um comando
O mesmo pipeline como script de linha de comando. Em um repositório real, isto fica atrás de `make reproduce`.

In [ ]:
script = '''
import json, sys, hashlib, numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score
c = json.load(open(sys.argv[sys.argv.index("--config") + 1]))
X, y = make_classification(n_samples=c["n_amostras"], weights=[0.95], random_state=c["semente"])
Xa, Xb, ya, yb = train_test_split(X, y, test_size=c["proporcao_teste"], stratify=y, random_state=c["semente"])
m = RandomForestClassifier(n_estimators=c["n_estimators"], class_weight=c["class_weight"], random_state=c["semente"]).fit(Xa, ya)
pred = (m.predict_proba(Xb)[:, 1] >= c["limiar"]).astype(int)
print(json.dumps({"recall": round(float(recall_score(yb, pred)), 6), "precisao": round(float(precision_score(yb, pred, zero_division=0)), 6)}))
'''
open("treinar.py", "w").write(script); json.dump(cfg.model_dump(), open("config.json", "w"))
# [SEU CÓDIGO AQUI] execute `python treinar.py --config config.json` em um subprocesso e capture a saída
print("linha de comando:", saida.strip(), "| manifesto:", m1["metricas"]); assert json.loads(saida) == m1["metricas"]

## Parte B — Uma decisão de arquitetura defensável

### Passo 8 — Matriz de decisão
Escolha do vector database (Semana 12). Critérios com peso e nota de 1 a 5 por opção. As notas abaixo são um exemplo: **substitua pelas suas**, com base no seu contexto.

In [ ]:
criterios = pd.DataFrame({
    "peso":     {"aproveita a infraestrutura atual": 0.25, "escala (>100M vetores)": 0.10, "filtros e consultas híbridas": 0.20,
                 "custo operacional": 0.20, "dados ficam no nosso ambiente": 0.15, "maturidade do ecossistema": 0.10},
    "pgvector": {"aproveita a infraestrutura atual": 5, "escala (>100M vetores)": 2, "filtros e consultas híbridas": 4, "custo operacional": 4, "dados ficam no nosso ambiente": 5, "maturidade do ecossistema": 4},
    "Qdrant":   {"aproveita a infraestrutura atual": 3, "escala (>100M vetores)": 4, "filtros e consultas híbridas": 4, "custo operacional": 4, "dados ficam no nosso ambiente": 5, "maturidade do ecossistema": 4},
    "Pinecone": {"aproveita a infraestrutura atual": 2, "escala (>100M vetores)": 5, "filtros e consultas híbridas": 3, "custo operacional": 3, "dados ficam no nosso ambiente": 1, "maturidade do ecossistema": 4},
})
opcoes = ["pgvector", "Qdrant", "Pinecone"]; assert abs(criterios["peso"].sum() - 1) < 1e-9
# [SEU CÓDIGO AQUI] pontuação ponderada de cada opção
print(pontuacao.round(2))

### Passo 9 — Sensitivity analysis
A decisão resiste a discordâncias sobre os pesos? Sorteie 5.000 vetores de pesos em torno dos originais (distribuição de Dirichlet) e conte quantas vezes cada opção vence.

In [ ]:
rng = np.random.default_rng(0)
# [SEU CÓDIGO AQUI] sorteie pesos, calcule o vencedor de cada sorteio e a fração de vitórias por opção
print(robustez.round(3))

### Passo 10 — O que inverteria a decisão?
Para cada critério, aumente o peso dele (reduzindo os demais proporcionalmente) até a opção vencedora mudar. Isso alimenta o "critério de revisão" do ADR.

In [ ]:
def peso_de_virada(criterio, passo=0.01):
    base = criterios["peso"]; campeao = pontuacao.index[0]
    for w in np.arange(base[criterio], 1.0, passo):
        # [SEU CÓDIGO AQUI] redistribua os pesos com `criterio` valendo w; se o vencedor mudar, devolva (w, novo vencedor)
    return None, None

for c in criterios.index:
    w, novo = peso_de_virada(c); print(f"{c:34s} | peso atual {criterios.loc[c, 'peso']:.2f} | vira em: {w} -> {novo}")

### Passo 11 — Gerar o ADR
Monte o Markdown do ADR a partir dos objetos. Os números vêm do código; o contexto e as consequências, de você.

In [ ]:
def gerar_adr(numero, titulo, contexto, consequencias, revisao):
    campeao = pontuacao.index[0]
    # [SEU CÓDIGO AQUI] monte a tabela de opções e devolva o texto completo do ADR

adr = gerar_adr(1, "Banco vetorial para o RAG corporativo",
                "[SEU TEXTO: o problema, o volume esperado, o que já existe na empresa, as restrições de dados]",
                "[SEU TEXTO: o que fica mais fácil, o que fica mais difícil, o que passa a ser obrigatório]",
                "[SEU TEXTO: que medição faria reabrir esta decisão]")
open("ADR-0001-banco-vetorial.md", "w", encoding="utf-8").write(adr); print(adr)

## Fechamento
Substitua as notas e os pesos pelos do seu contexto, preencha os três textos e salve o ADR em `docs/`. Depois escreva o RFC "Como padronizaremos os A/B tests" usando o modelo em `docs/templates/RFC-template.md`.